In [1]:
from itertools import zip_longest
import gc

import numpy as np

"""
ToyNetwork 재분석
    화성~서울 네트워크(서울방향만)
    집계시간 : 5, 10, 15분
    분석시간 1800~13200
    램프 ALL
"""
import pandas as pd

import os

# FIX 값 모음
###################################################################################################################

path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\Test"

# 한번에 처리할 .mer파일 갯수
num_mer = 5

# 집계시간(분)
min_time = 5 #5,10,15

# 집계시간(초)
sec_time = 60 * min_time

#대/시 환산
revision_time = 60/min_time

start_interval = 1800
end_interval = 13200

weights = {
    "w1" : 0.1,
    "w2" : 0.1,
    "w3" : 0.1,
    "w4" : 0.1,
    "w5" : 0.3,
    "w6" : 0.3
}

vehicle_types = [100, 300, 630, 640, 650]

# 램프 간섭 영향률
###################################################################################################################
# 램프 전 본선 검지기(램프 간섭 영향률)
before_ramp = [70, 117, 124, 186, 215]

# 램프 후 본선 검지기(램프 간섭 영향률)
after_ramp = [74, 121, 127, 190, 221]

# 유입램프 검지기(램프 간섭 영향률)
input_ramp = [902, 904]

# 유출램프 검지기(램프 간섭 영향률)
output_ramp = [901, 903, 905]

# 진출 정상성(진입)(진출 원활률)
enter_line = [23, 121, 190]

# 유입램프 바로 뒤 본선 검지기(진출 원활률)(램프 간섭 영향률)
input_main_ramp = [121, 190]

# 유출램프 바로 앞 본선 검지기(진출 원활률)(램프 간섭 영향률)
output_main_ramp = [73, 126, 220]

# 3차로 검지기
three_lane = [71, 72, 73, 119, 120, 125, 126, 188, 189, 216, 217, 218, 219, 220]
###################################################################################################################

# 함수 모음
###################################################################################################################

# 속도 변화율
def speed_mean(original_df):
    copy_df = original_df.copy()

    # 램프 검지기 제외
    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    # TimeGroup, New_Measurement별 그룹화 및 속도 평균
    speed_mean_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
          .agg(V_mean=("v[km/h]", "mean"), V_count=("v[km/h]", "count"))
          .reset_index()
    )
    speed_mean_df["V_next"] = speed_mean_df.groupby("TimeGroup")["V_mean"].shift(-1)
    cols=["V_mean", "V_next"]
    speed_mean_df[cols] = speed_mean_df[cols].fillna(0)
    speed_mean_df["delta_V"] = np.where(speed_mean_df["V_mean"] == 0,
        0,
        (speed_mean_df["V_next"] - speed_mean_df["V_mean"]) / speed_mean_df["V_mean"]
    )


    return speed_mean_df

# 밀도 변화율
def density_mean(speed_df):
    density_mean_df  = speed_df.copy()

    density_mean_df["K"] = np.where(
        density_mean_df["V_mean"] == 0,
        0,
        density_mean_df["V_count"] * revision_time / density_mean_df["V_mean"]
    )

    density_mean_df["K_next"] = density_mean_df.groupby("TimeGroup")["K"].shift(-1)
    cols=["K", "K_next"]
    density_mean_df[cols] = density_mean_df[cols].fillna(0)
    density_mean_df["delta_K"] = np.where(density_mean_df["K"] == 0,
        0,
        (density_mean_df["K_next"] - density_mean_df["K"]) / density_mean_df["K"]
    )
    return density_mean_df

# 중차량 혼입률
def heavy_rate(original_df):
    copy_df = original_df.copy()

    heavy_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .agg(
            heavy_count=("Vehicle type", lambda x: x.isin([630, 640, 650]).sum()),
            total_count=("Vehicle type", "count")
        )
        .reset_index()
    )

    heavy_df["rate"] = np.where(
        heavy_df["total_count"] == 0,
        0,
        heavy_df["heavy_count"] / heavy_df["total_count"]
    )

    return heavy_df

# 동적 포화도
def entry_saturation(original_df):
    copy_df = original_df.copy()

    # 실측용량 C(2차로 4400)
    entry_saturation_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .size()
        .reset_index(name="entry_volume")  # 차량 수를 entry_volume이라는 컬럼명으로
    )

    # 행별 capacity 설정
    entry_saturation_df["capacity"] = np.where(
        entry_saturation_df["New_Measurement"].isin(three_lane),
        6600,   # 3차로
        4400    # 2차로
    )

    entry_saturation_df["Phi_진입"] = entry_saturation_df["entry_volume"] * revision_time / entry_saturation_df["capacity"]

    return entry_saturation_df

# 램프 간섭 영향률
def rfr_rate(original_df):
    copy_df = original_df.copy()
    copy_df["TimeGroup"] = copy_df["TimeGroup"].astype(str)
    main_results=[]

    for i, (before, after) in enumerate(zip(before_ramp, after_ramp)):
        q_before = (copy_df[copy_df["New_Measurement"] == before] # 70, 117, 124, 186, 215, 312, 342, 403, 412, 460
                    .groupby("TimeGroup")
                    .size()
                    .reset_index(name="q_before"))

        q_after = (copy_df[copy_df["New_Measurement"] == after] # 74, 121, 127, 190, 221, 317, 345, 406, 416, 465
                   .groupby("TimeGroup")
                   .size()
                   .reset_index(name="q_after"))

        merged = q_before.merge(q_after, on="TimeGroup", how="outer").fillna(0)
        merged["before_ramp"] =  before
        merged["after_ramp"] =  after
        merged["Qm"] = (merged["q_before"] + merged["q_after"]) / 2
        main_results.append(merged)


    ramp_results = []
    for input_, output_ in zip_longest(input_ramp, output_ramp):
        if output_ is not None:
            q_out = (copy_df[copy_df["New_Measurement"] == output_] # 901, 903, 905
                     .groupby("TimeGroup").size().reset_index(name="q_out"))
            q_out["New_Measurement"] = output_
            ramp_results.append(q_out)
        if input_ is not None:
            q_in = (copy_df[copy_df["New_Measurement"] == input_] # 902, 904
                    .groupby("TimeGroup").size().reset_index(name="q_in"))
            q_in["New_Measurement"] = input_
            ramp_results.append(q_in)


    input_queue = input_main_ramp.copy()
    output_queue = output_main_ramp.copy()
    rfr_list = []

    for i in range(min(len(main_results), len(ramp_results))):
        main_df = main_results[i]
        ramp_df = ramp_results[i]

        rfr_df = pd.merge(main_df, ramp_df, on="TimeGroup", how="outer").fillna(0)
        display("rfr_df : ", rfr_df)
        # 기본값 초기화
        rfr_df["IR_in"] = 0
        rfr_df["IR_out"] = 0

        # q_in 있을 때 (유입)
        if "q_in" in rfr_df.columns:
            rfr_df["IR_in"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_in"] / rfr_df["Qm"]
            )
            if input_queue:  # 남은 게 있으면 하나 꺼냄
                current_input = input_queue.pop(0)
                rfr_df["New_Measurement"] = current_input

        # q_out 있을 때 (유출)
        if "q_out" in rfr_df.columns:
            rfr_df["IR_out"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_out"] / rfr_df["Qm"]
            )
            if output_queue:
                current_output = output_queue.pop(0)
                rfr_df["New_Measurement"] = current_output

        rfr_df["RFR"] = rfr_df["IR_in"] + rfr_df["IR_out"]

        rfr_list.append(rfr_df)

    final_rfr_df = pd.concat(rfr_list, ignore_index=True)
    display("final_rfr_df : ", final_rfr_df)
    # -----------------------------
    # 특정 검지기에만 RFR 반영
    # -----------------------------
    target_measurements = input_main_ramp + output_main_ramp
    all_measurements = copy_df["New_Measurement"].unique()

    expanded_df_list = []

    base_rfr_df = final_rfr_df.copy()

    for m in all_measurements:
        if m in target_measurements:
            temp = base_rfr_df[base_rfr_df["New_Measurement"] == m].copy()
        else:
            temp = base_rfr_df[["TimeGroup"]].drop_duplicates().copy()
            temp["New_Measurement"] = m
            temp["RFR"] = 0

        expanded_df_list.append(temp)

    final_rfr_df = pd.concat(expanded_df_list, ignore_index=True)
    final_rfr_df = final_rfr_df.sort_values(by=["TimeGroup", "New_Measurement"]).reset_index(drop=True)
    final_rfr_df = final_rfr_df[["TimeGroup", "New_Measurement", "RFR"]]
    final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)
    return final_rfr_df


# 진출 원활율- output_main_ramp
def output_normality(original_df):
    copy_df = original_df.copy()

    normality_list = []

    # 여러 진입/진출 쌍 처리
    for enter, exit_ramp, exit_main in zip(enter_line, output_ramp, output_main_ramp):

        entry_df = copy_df[copy_df["New_Measurement"] == enter][["New_Measurement", "VehNo", "t(Entry)"]]

        exit_df  = copy_df[copy_df["New_Measurement"] == exit_ramp][["New_Measurement", "VehNo", "t(Entry)"]]

        # 차량 번호별 최소 통과시각
        entry_first = (
            entry_df.groupby("VehNo")["t(Entry)"].min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_entry"})
        )

        exit_first = (
            exit_df.groupby("VehNo")["t(Entry)"].min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_exit"})
        )

        # 진입-진출 매칭 후 지연시간 계산
        merged = pd.merge(entry_first, exit_first, on="VehNo", how="inner")
        merged["delay_sec"] = merged["t_exit"] - merged["t_entry"]
        merged = merged[merged["delay_sec"] >= 0]  # 음수 제거

        # 중간값 기반 시간지연 bin 계산
        if len(merged) and np.isfinite(np.nanmedian(merged["delay_sec"])):
            lag_bins = int(round(np.nanmedian(merged["delay_sec"]) / sec_time))
        else:
            lag_bins = 0

        # 진입/진출 카운트 집계
        entry_count = (
            copy_df[copy_df["New_Measurement"] == enter]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_in")
        )

        exit_count = (
            copy_df[copy_df["New_Measurement"] == exit_ramp]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_out")
        )

        # 병합 후 지연만큼 shift
        merged_counts = pd.merge(entry_count, exit_count, on="TimeGroup", how="left")


        merged_counts["Q_out_shift"] = merged_counts["Q_out"].shift(-lag_bins)


        # F(outrate)
        #merged_counts["F(outrate)"] = (merged_counts["Q_out_shift"] / merged_counts["Q_in"]).fillna(0)

        merged_counts["F(outrate)"] = np.where(
            merged_counts["Q_in"] == 0,
            0,
            merged_counts["Q_out_shift"] / merged_counts["Q_in"]
        )

        merged_counts["New_Measurement"] = exit_main  # 진출 지점에 부여

        normality_list.append(merged_counts)


    # 모든 진출 램프 결과 병합
    final_df = pd.concat(normality_list, ignore_index=True)


    # 전체 검지기 확장
    all_measurements = copy_df["New_Measurement"].unique()
    expanded_list = []

    for m in all_measurements:
        if m in output_main_ramp:
            temp = final_df[final_df["New_Measurement"] == m].copy()
        else:
            temp = final_df[["TimeGroup"]].drop_duplicates().copy()
            temp["New_Measurement"] = m
            temp["F(outrate)"] = 0
        expanded_list.append(temp)

    final_df = pd.concat(expanded_list, ignore_index=True)
    final_df = final_df.sort_values(by=["TimeGroup", "New_Measurement"]).reset_index(drop=True)

    return final_df


def calculate_stvm(speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df):

    # TimeGroup 기준으로  Merge
    merged_df = (
        speed_df[["TimeGroup", "New_Measurement", "delta_V"]]
        .merge(density_df[["TimeGroup", "New_Measurement", "delta_K"]], on=["TimeGroup", "New_Measurement"])
        .merge(heavy_df[["TimeGroup", "New_Measurement", "rate"]], on=["TimeGroup", "New_Measurement"])
        .merge(entry_saturation_df[["TimeGroup", "New_Measurement", "Phi_진입"]], on=["TimeGroup", "New_Measurement"])
        .merge(rfr_df[["TimeGroup", "New_Measurement", "RFR"]], on=["TimeGroup", "New_Measurement"])
        .merge(normality_df[["TimeGroup", "New_Measurement", "F(outrate)"]], on=["TimeGroup", "New_Measurement"])
    )

    merged_df["STVM"] = (
        weights["w1"] * merged_df["delta_V"] +
        weights["w2"] * merged_df["delta_K"] +
        weights["w3"] * merged_df["rate"] +
        weights["w4"] * merged_df["Phi_진입"] +
        weights["w5"] * merged_df["RFR"] +
        weights["w6"] * merged_df["F(outrate)"]
    )
    merged_df.replace([np.inf, -np.inf], 0, inplace=True)
    merged_df.fillna(0, inplace=True)
    merged_df = modify_frame(merged_df)
    return merged_df

def calc_z(df):
    copy_df = df.copy()
    if copy_df.empty:
        return copy_df

    # 검지기별 평균
    stvm_avg_df = (
        copy_df
        .groupby("New_Measurement")["STVM"]
        .mean()
        .reset_index(name="STVM_MEAN")
    )



    avg_stvm = stvm_avg_df["STVM_MEAN"].mean()
    std_stvm = stvm_avg_df["STVM_MEAN"].std(ddof=0)

    stvm_avg_df["Z-변환"] = (stvm_avg_df["STVM_MEAN"] - avg_stvm) / std_stvm

    z_max = stvm_avg_df["Z-변환"].max()
    z_min = stvm_avg_df["Z-변환"].min()
    stvm_avg_df["환산점수"] = stvm_avg_df["Z-변환"].apply(lambda z: z_to_score(z, z_min, z_max))

    return stvm_avg_df

def calculate_z_score(stvm_df):
    copy_df = stvm_df.copy()

    # 구간 나누기
    group1 = copy_df[copy_df["New_Measurement"].between(1, 265)].copy()

    group1 = calc_z(group1)

    merged = group1.sort_values(by="New_Measurement")
    #save_to_excel(merged, folder_path, "환산점수 원시데이터", i)

    #stvm_df = pd.pivot(merged, index="TimeGroup", columns= "New_Measurement", values="환산점수")

    return merged

def modify_frame(df):
    modify_df = df.copy()
    modify_df["StartTime"] = modify_df["TimeGroup"].str.split("~").str[0].astype(int)
    modify_df = modify_df[(modify_df["StartTime"] >=start_interval) &(modify_df["StartTime"] < end_interval)]
    modify_df = modify_df[~modify_df["New_Measurement"].isin([266, 901, 902, 903, 904, 905])]
    modify_df = modify_df.reset_index(drop=True)
    return modify_df


def variable_timegroup_avg(stvm_df):
    copy_df = stvm_df.copy()
    variable_time_df = copy_df.groupby("TimeGroup")[["delta_V", "delta_K", "rate", "Phi_진입", "RFR", "F(outrate)"]].mean()
    return variable_time_df

def variable_total_avg(variable_df):
    variable_total_df = pd.DataFrame([variable_df.mean(numeric_only=True)])
    return variable_total_df

def speed_density_avg(density_df):
    copy_df = density_df.copy()
    avg_df = modify_frame(copy_df)
    avg_df = pd.DataFrame([avg_df.mean(numeric_only=True)])
    avg_df = avg_df[["V_mean", "K"]]
    return avg_df

def pivot_table(df, value, preprocess=None):
    copy_df = df.copy()
    if preprocess :
        copy_df = preprocess(copy_df)
    copy_df = copy_df.pivot(index="TimeGroup", columns="New_Measurement", values=value)

    copy_df = (
        copy_df
        .assign(_t=lambda x: x.index.astype(str).str.split("~").str[0].astype(int))
        .sort_values("_t")
        .drop(columns="_t")
    )
    copy_df = copy_df.fillna(0)
    return copy_df

def stvm_color(val):
    if pd.isna(val):
        return ""
    elif val <= 0:
        return "background-color: #00B050" # 초록
    else:
        return "background-color: #FFC000"  # 주황색


def weighted_avg_speed(original_df):
    copy_df = original_df.copy()
    # TimeGroup, New_Measurement별 그룹화 및 속도 평균
    speed_mean_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement", "Vehicle type"])
          .agg(V_mean=("v[km/h]", "mean"), V_count=("v[km/h]", "count"))
          .reset_index()
    )
    speed_mean_df["std_group"] = speed_mean_df.groupby(["TimeGroup", "New_Measurement"])["V_mean"].transform(lambda s: s.std(ddof=0))
    speed_mean_df["cv"] = speed_mean_df["std_group"] / speed_mean_df["V_mean"]
    speed_mean_df["w"] = 1 / speed_mean_df["cv"]
    speed_mean_df["w*v"] = speed_mean_df["w"] * speed_mean_df["V_mean"]

    weighted_result = (
        speed_mean_df.groupby(["TimeGroup","New_Measurement"])
          .apply(lambda g: g["w*v"].sum() / g["w"].sum())
          .reset_index(name="Weighted_Avg_Speed")
    )

    return weighted_result

def save_to_excel(excel_df, folder_path, file_name, i, color=False):
        excel_folder_path = os.path.join(folder_path, file_name)
        os.makedirs(excel_folder_path, exist_ok=True)
        excel_file_name = f"{file_name}_{i+1}.xlsx"
        excel_file_path = os.path.join(excel_folder_path, excel_file_name)

        if color:
            styled = excel_df.style.applymap(stvm_color)
            styled.to_excel(excel_file_path, engine="openpyxl")
        else:
            excel_df.to_excel(excel_file_path, index=True)

        print(f"{excel_file_name} 생성 완료")

def z_to_score(z, z_min, z_max):
    if 1.645 <= z <= z_max:
        return 50 + ((95 + 5 * ((z - 1.645) / (z_max - 1.645))) * 0.5)
    elif 1.282 <= z < 1.645:
        return 50 + ((90 + 5 * ((z - 1.282) / (1.645 - 1.282))) * 0.5)
    elif 1.038 <= z < 1.282:
        return 50 + ((85 + 5 * ((z - 1.038) / (1.282 - 1.038))) * 0.5)
    elif 0.842 <= z < 1.038:
        return 50 + ((80 + 5 * ((z - 0.842) / (1.038 - 0.842))) * 0.5)
    elif 0.676 <= z < 0.842:
        return 50 + ((75 + 5 * ((z - 0.676) / (0.842 - 0.676))) * 0.5)
    elif 0.526 <= z < 0.676:
        return 50 + ((70 + 5 * ((z - 0.526) / (0.676 - 0.526))) * 0.5)
    elif 0.387 <= z < 0.526:
        return 50 + ((65 + 5 * ((z - 0.387) / (0.526 - 0.387))) * 0.5)
    elif 0.255 <= z < 0.387:
        return 50 + ((60 + 5 * ((z - 0.255) / (0.387 - 0.255))) * 0.5)
    elif -0.255 <= z < 0.255:
        return 50 + ((40 + 5 * ((z + 0.255) / (0.255 + 0.255))) * 0.5)
    elif -0.387 <= z < -0.255:
        return 50 + ((35 + 5 * ((z + 0.387) / (-0.255 + 0.387))) * 0.5)
    elif -0.526 <= z < -0.387:
        return 50 + ((30 + 5 * ((z + 0.526) / (-0.387 + 0.526))) * 0.5)
    elif -0.676 <= z < -0.526:
        return 50 + ((25 + 5 * ((z + 0.676) / (-0.676 + 0.842))) * 0.5)
    elif -0.842 <= z < -0.676:
        return 50 + ((20 + 5 * ((z + 0.842) / (-0.676 + 0.842))) * 0.5)
    elif -1.038 <= z < -0.842:
        return 50 + ((15 + 5 * ((z + 1.038) / (-0.842 + 1.038))) * 0.5)
    elif -1.282 <= z < -1.038:
        return 50 + ((10 + 5 * ((z + 1.282) / (-1.038 + 1.282))) * 0.5)
    elif -1.645 <= z < -1.282:
        return 50 + ((5 + 5 * ((z + 1.645) / (-1.282 + 1.645))) * 0.5)
    elif z_min <= z < -1.645:
        return 50 + ((0 + 5 * ((z + z_min) / (-1.645 + z_min))) * 0.5)
    else:
        return np.nan
###################################################################################################################

folder_path = path
mer_list = sorted([file for file in os.listdir(folder_path) if file.endswith(".mer")])

for start in range(0, len(mer_list), num_mer):
    batch_files = mer_list[start:start + num_mer]
    print(f"\n===== {start+1} ~ {start+len(batch_files)} 번째 파일 처리 =====")

    i = start // num_mer

    speed_list = []
    density_list = []
    heavy_list = []
    entry_list = []
    rfr_list_all = []
    normality_list_all = []
    stvm_list = []

    for index, mer_file in enumerate(batch_files):
        print("작업파일 :", mer_file)

        with open(os.path.join(folder_path, mer_file), "r", encoding="utf-8", errors="ignore") as file:
            lines = file.readlines()

        data_start_idx = None
        for j, line in enumerate(lines):
            if "Measurem." in line:
                data_start_idx = j
                break

        # 데이터프레임 생성
        if data_start_idx is not None:

            # 컬럼명 추출 및 공백 제거
            columns = [col.strip() for col in lines[data_start_idx].strip().split(";")]

            # 데이터 부분 추출 및 가공
            data_lines = lines[data_start_idx + 1:]  # 컬럼명 제외, 데이터 부분
            data = [line.strip().split(";") for line in data_lines if line.strip()]

            # 데이터프레임 생성
            df = pd.DataFrame(data, columns=columns)

            # 컬럼 내부 데이터 정수형 변환
            df = df.apply(pd.to_numeric, errors="coerce")

            original_df = df[(df["t(Entry)"] != -1.00)].reset_index(drop=True)

            #불필요 컬럼 제거
            original_df.drop(columns=["b[m/s2]", "tQueue", "Occ", "Pers"], inplace=True, errors="ignore")

            original_df["New_Measurement"] = original_df["Measurem."] % 1000

            bins = np.arange(start_interval, end_interval+1, sec_time)
            labels = [f"{start}~{start+sec_time}" for start in bins[:-1]]  # 구간 라벨링

            # 구간 나누기 및 컬럼 추가
            original_df["TimeGroup"] = pd.cut(original_df["t(Entry)"], bins=bins, labels=labels, right=False)

            # 가감속 차로 부분 검지기 제거
            #original_df = original_df[~original_df["Measurem."].between(30000, 39999)]

            # 속도변화율
            speed_df = speed_mean(original_df)
            speed_list.append(speed_df)


            # 밀도변화율
            density_df = density_mean(speed_df)
            density_list.append(density_df)

            # 중차량 혼입률
            heavy_df = heavy_rate(original_df)
            heavy_list.append(heavy_df)

            # 동적 포화도
            entry_saturation_df = entry_saturation(original_df)
            entry_list.append(entry_saturation_df)

            # 램프 간섭 영향률
            rfr_df = rfr_rate(original_df)
            rfr_list_all.append(rfr_df)

            # 진출 원활율
            normality_df = output_normality(original_df)
            normality_list_all.append(normality_df)

            # STVM 계산
            stvm_df = calculate_stvm(speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df)
            stvm_list.append(stvm_df)

    print("시드 평균 계산 중...")

    merged_speed = pd.concat(speed_list)
    merged_density = pd.concat(density_list)
    merged_heavy = pd.concat(heavy_list)
    merged_entry = pd.concat(entry_list)
    merged_rfr = pd.concat(rfr_list_all)
    merged_normality = pd.concat(normality_list_all)
    merged_stvm = pd.concat(stvm_list)

    # 각 지표 평균
    speed_avg = (
        merged_speed
        .groupby(["TimeGroup", "New_Measurement"])
        [["delta_V"]]
        .mean()
        .reset_index()
    )

    density_avg = (
        merged_density
        .groupby(["TimeGroup", "New_Measurement"])
        [["delta_K"]]
        .mean()
        .reset_index()
    )

    heavy_avg = (
        merged_heavy
        .groupby(["TimeGroup", "New_Measurement"])
        [["rate"]]
        .mean()
        .reset_index()
    )

    entry_avg = (
        merged_entry
        .groupby(["TimeGroup", "New_Measurement"])
        [["Phi_진입"]]
        .mean()
        .reset_index()
    )

    rfr_avg = (
        merged_rfr
        .groupby(["TimeGroup", "New_Measurement"])
        [["RFR"]]
        .mean()
        .reset_index()
    )

    normality_avg = (
        merged_normality
        .groupby(["TimeGroup", "New_Measurement"])
        [["F(outrate)"]]
        .mean()
        .reset_index()
    )

    stvm_avg = (
        merged_stvm
        .groupby(["TimeGroup", "New_Measurement"])
        .mean(numeric_only=True)
        .reset_index()
    )
    #save_to_excel(speed_avg, folder_path, "속도변화량", i)
    #save_to_excel(density_avg, folder_path, "밀도변화량", i)
    #save_to_excel(stvm_avg, folder_path, "STVM", i)

    # Z-Score 계산
    z_score_df = calculate_z_score(stvm_avg)
    #save_to_excel(z_score_df, folder_path, f"환산점수", i)

    # STVM 피봇
    stvm_pivot_df = pivot_table(stvm_avg, "STVM", preprocess=modify_frame)
    #save_to_excel(stvm_pivot_df, folder_path, f"지표합산값", i, color=False)

    # 속도값 피봇
    #speed_pivot_df = pivot_table(speed_df, "V_mean", preprocess=modify_frame)
    #display("speed_pivot_df : ", speed_pivot_df)
    #save_to_excel(speed_pivot_df, folder_path, "속도 피봇", i)

    # 지표별 구성값(속도변화값)
    #speed_pivot_df = pivot_table(speed_df, "delta_V", preprocess=modify_frame)
    #display("speed_pivot_df : ", speed_pivot_df)
    #save_to_excel(speed_pivot_df, folder_path, "속도변화값", i)

    # 지표별 구성값(밀도변화값)
    #density_pivot_df = pivot_table(density_df, "delta_K", preprocess=modify_frame)
    #display("density_pivot_df : ", density_pivot_df)
    #save_to_excel(density_pivot_df, folder_path, "밀도변화값", i)

    # 지표별 구성값(중차량혼입률)
    #heavy_pivot_df = pivot_table(heavy_df, "rate", preprocess=modify_frame)
    #display("heavy_pivot_df : ", heavy_pivot_df)
    #save_to_excel(heavy_pivot_df, folder_path, "중차량혼입률", i)

    # 지표별 구성값(동적포화도)
    #entry_saturation_pivot_df = pivot_table(entry_saturation_df, "Phi_진입", preprocess=modify_frame)
    #display("entry_saturation_pivot_df : ", entry_saturation_pivot_df)
    #save_to_excel(entry_saturation_pivot_df, folder_path, "동적포화도", i)

    # 지표별 구성값(램프 간섭 영향률)
    #rfr_pivot_df = pivot_table(rfr_df, "RFR", preprocess=modify_frame)
    #display("rfr_pivot_df : ", rfr_pivot_df)
    #save_to_excel(rfr_pivot_df, folder_path, "램프간섭영향률", i)

    # 지표별 구성값(진출 원활률)
    #normality_pivot_df = pivot_table(normality_df, "F(outrate)", preprocess=modify_frame)
    #display("normality_pivot_df : ", normality_pivot_df)
    #save_to_excel(normality_pivot_df, folder_path, "진출원활률", i)

    # 메모리 정리
    #del df, original_df, speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df, stvm_df, z_score_df
    gc.collect()


===== 1 ~ 1 번째 파일 처리 =====
작업파일 : 화성~서울(유고2차로_서비스E_유고5분)_260311_001.mer


'rfr_df : '

,TimeGroup,q_before,q_after,before_ramp,after_ramp,Qm,q_out,New_Measurement
0,10200~10500,319,279,70,74,299.0,46,901
1,10500~10800,259,212,70,74,235.5,37,901
2,10800~11100,315,262,70,74,288.5,43,901
3,11100~11400,315,271,70,74,293.0,42,901
4,11400~11700,314,302,70,74,308.0,41,901
5,11700~12000,272,213,70,74,242.5,46,901
6,12000~12300,322,267,70,74,294.5,47,901
7,12300~12600,304,276,70,74,290.0,43,901
8,12600~12900,336,278,70,74,307.0,50,901
9,12900~13200,271,236,70,74,253.5,46,901


'rfr_df : '

,TimeGroup,q_before,q_after,before_ramp,after_ramp,Qm,q_in,New_Measurement
0,10200~10500,213,238,117,121,225.5,30,902
1,10500~10800,284,330,117,121,307.0,25,902
2,10800~11100,222,251,117,121,236.5,32,902
3,11100~11400,282,264,117,121,273.0,24,902
4,11400~11700,245,312,117,121,278.5,19,902
5,11700~12000,249,276,117,121,262.5,14,902
6,12000~12300,268,260,117,121,264.0,29,902
7,12300~12600,251,269,117,121,260.0,22,902
8,12600~12900,290,327,117,121,308.5,18,902
9,12900~13200,287,318,117,121,302.5,20,902


...

'rfr_df : '

,TimeGroup,q_before,q_after,before_ramp,after_ramp,Qm,q_out,New_Measurement
0,10200~10500,244,226,215,221,235.0,26,905
1,10500~10800,237,214,215,221,225.5,36,905
2,10800~11100,249,200,215,221,224.5,34,905
3,11100~11400,212,180,215,221,196.0,34,905
4,11400~11700,241,216,215,221,228.5,29,905
5,11700~12000,264,214,215,221,239.0,42,905
6,12000~12300,241,228,215,221,234.5,34,905
7,12300~12600,232,191,215,221,211.5,34,905
8,12600~12900,262,212,215,221,237.0,46,905
9,12900~13200,271,241,215,221,256.0,40,905


'final_rfr_df : '

,TimeGroup,q_before,q_after,before_ramp,after_ramp,Qm,q_out,New_Measurement,IR_in,IR_out,RFR,q_in
0,10200~10500,319,279,70,74,299.0,46.0,73,0.0,0.153846,0.153846,NaN
1,10500~10800,259,212,70,74,235.5,37.0,73,0.0,0.157113,0.157113,NaN
2,10800~11100,315,262,70,74,288.5,43.0,73,0.0,0.149047,0.149047,NaN
3,11100~11400,315,271,70,74,293.0,42.0,73,0.0,0.143345,0.143345,NaN
4,11400~11700,314,302,70,74,308.0,41.0,73,0.0,0.133117,0.133117,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
185,8700~9000,210,168,215,221,189.0,33.0,220,0.0,0.174603,0.174603,NaN
186,9000~9300,261,222,215,221,241.5,37.0,220,0.0,0.153209,0.153209,NaN
187,9300~9600,294,251,215,221,272.5,41.0,220,0.0,0.150459,0.150459,NaN
188,9600~9900,212,179,215,221,195.5,30.0,220,0.0,0.153453,0.153453,NaN


시드 평균 계산 중...
